In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.listdir('/content/drive/MyDrive/OULAD_processed')

In [ ]:
import pandas as pd
OUTPUT_PATH = '/content/drive/MyDrive/OULAD_processed/'

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    roc_auc_score, classification_report,
    confusion_matrix, roc_curve
)
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import cdist

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

# ── Constantes ───────────────────────────────────────────────
student_id_cols    = ['id_student', 'code_module', 'code_presentation']
K_OPTIMAL          = 4
HIDDEN_SIZE        = 64
CHECKPOINTS        = [3, 7, 12]
BATCH_SIZE         = 64
N_EPOCHS           = 50
LR                 = 1e-3
SEED               = 42


torch.manual_seed(SEED)
np.random.seed(SEED)


# ════════════════════════════════════════════════════════════
# ETAPE 1 — CHARGEMENT DES DONNÉES
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("ÉTAPE 1 — Chargement")
print("=" * 60)

X_seq       = np.load(OUTPUT_PATH + 'X_seq.npy')
y_arr       = np.load(OUTPUT_PATH + 'y_arr.npy')
mask_seq    = np.load(OUTPUT_PATH + 'mask_seq.npy')
final_df    = pd.read_csv(OUTPUT_PATH + 'oulad_final.csv')
final_df_wcdmacp = pd.read_csv(OUTPUT_PATH + 'oulad_final_with_code_module_and_code_presentation.csv')
weekly_combined = pd.read_csv(OUTPUT_PATH + 'oulad_weekly_v2.csv')
X_train = np.load(OUTPUT_PATH + 'X_train.npy')
X_val   = np.load(OUTPUT_PATH + 'X_val.npy')
X_test  = np.load(OUTPUT_PATH + 'X_test.npy')

y_train = np.load(OUTPUT_PATH + 'y_train.npy')
y_val   = np.load(OUTPUT_PATH + 'y_val.npy')
y_test  = np.load(OUTPUT_PATH + 'y_test.npy')

mask_train = np.load(OUTPUT_PATH + 'mask_train.npy')
mask_val   = np.load(OUTPUT_PATH + 'mask_val.npy')
mask_test  = np.load(OUTPUT_PATH + 'mask_test.npy')


with open(OUTPUT_PATH + 'student_ids.pkl', 'rb') as f:
    student_ids = pickle.load(f)

with open(OUTPUT_PATH + 'scaler_seq.pkl', 'rb') as f:
    scaler = pickle.load(f)

N, T, F = X_train.shape
print(f"X_seq     : {X_seq.shape}")
print(f"y_arr     : {y_arr.shape}")
print(f"mask_seq  : {mask_seq.shape}")
print(f"final_df  : {final_df.shape}")
print(f"Taux at_risk : {y_arr.mean()*100:.1f}%")


print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_val   : {y_val.shape}")
print(f"y_test  : {y_test.shape}")
print(f"at_risk Train : {y_train.mean()*100:.1f}%")
print(f"at_risk Val   : {y_val.mean()*100:.1f}%")
print(f"at_risk Test  : {y_test.mean()*100:.1f}%")


# ════════════════════════════════════════════════════════════
# ETAPE 2 — DATASET + DATALOADER
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ÉTAPE 2 — Dataset & DataLoader")
print("=" * 60)

class OULADDataset(Dataset):
    def __init__(self, X, y, mask):
        self.X    = torch.tensor(X,    dtype=torch.float32)
        self.y    = torch.tensor(y,    dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.mask[idx]


train_loader = DataLoader(
    OULADDataset(X_train, y_train, mask_train),
    batch_size=BATCH_SIZE, shuffle=True
)

val_loader = DataLoader(
    OULADDataset(X_val, y_val, mask_val),
    batch_size=BATCH_SIZE, shuffle=False
)

test_loader = DataLoader(
    OULADDataset(X_test, y_test, mask_test),
    batch_size=BATCH_SIZE, shuffle=False
)


# ════════════════════════════════════════════════════════════
# ETAPE 3 — ARCHITECTURE GRU
# ════════════════════════════════════════════════════════════

In [ ]:
print("\n" + "=" * 60)
print("ÉTAPE 3 — Architecture GRU")
print("=" * 60)

class GRU_Pipeline(nn.Module):
    """
    GRU unidirectionnel qui produit simultanément :
    - Embeddings 64D 
    - Risk score 0-1

    Le masque est utilisé pour ignorer les semaines de padding.
    """

    def __init__(self, input_size, hidden_size=64,
                 n_layers=2, dropout=0.3):
        super().__init__()

        self.hidden_size = hidden_size
        self.n_layers    = n_layers

        # ── GRU ──────────────────────────────────────────
        self.gru = nn.GRU(
            input_size    = input_size,
            hidden_size   = hidden_size,
            num_layers    = n_layers,
            batch_first   = True,
            dropout       = dropout if n_layers > 1 else 0.0,
            bidirectional = False
        )

        # ── Batch Normalization ───────────────────────────
        self.bn = nn.BatchNorm1d(hidden_size)

        # ── Tête 1 : Embedding pour K-Means ─────────
        self.embedding_head = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
        )

        # ── Tête 2 : Risk score pour EWS ────────────
        self.risk_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x, mask=None):
        """
        x    : (batch, 38, n_features)
        mask : (batch, 38) — 1=vraie semaine, 0=padding
        """
        gru_out, hidden = self.gru(x)
        # hidden : (n_layers, batch, hidden_size)

        if mask is not None:
            # Utiliser le dernier timestep RÉEL de chaque séquence
            # au lieu du dernier timestep fixe (qui peut être du padding)
            lengths = mask.sum(dim=1).long()           # (batch,)
            lengths = torch.clamp(lengths - 1, min=0)  # index du dernier réel

            batch_size = x.size(0)
            # Récupérer l'output GRU au dernier timestep réel
            last_hidden = gru_out[
                torch.arange(batch_size, device=x.device),
                lengths
            ]  # (batch, hidden_size)
        else:
            # Sinon prendre simplement le dernier timestep
            last_hidden = hidden[-1]  # (batch, hidden_size)

        last_hidden = self.bn(last_hidden)

        # Tête 1 : embedding pour clustering
        embedding  = self.embedding_head(last_hidden)   # (batch, 64)

        # Tête 2 : score de risque
        risk_score = self.risk_head(last_hidden)         # (batch, 1)

        return embedding, risk_score

    def get_checkpoint_output(self, x, mask, checkpoint_week):
        """
        Produit embedding + risk_score en regardant
        uniquement les semaines 0 → checkpoint_week.
        Utilisé pour le Dynamic Profiling P-CEA.
        """
        # Couper la séquence au checkpoint
        x_partial    = x[:, :checkpoint_week + 1, :]
        mask_partial = mask[:, :checkpoint_week + 1] if mask is not None else None

        return self.forward(x_partial, mask_partial)


    def get_temporal_embeddings(self, x, mask=None):
      """
      Retourne les embeddings temporels GRU :
      x : (batch, 38, n_features)
      output : (batch, 38, 64)
      """
      gru_out, hidden = self.gru(x)  # (batch, 38, hidden_size)

      B, T, H = gru_out.shape

      # Appliquer BatchNorm timestep par timestep
      gru_out_flat = gru_out.reshape(B * T, H)
      gru_out_bn = self.bn(gru_out_flat)
      gru_out_bn = gru_out_bn.reshape(B, T, H)

      # Appliquer embedding_head à chaque timestep
      temporal_embeddings = self.embedding_head(gru_out_bn)  # (B, T, 64)

      # Mettre à zéro les semaines padding
      if mask is not None:
        temporal_embeddings = temporal_embeddings * mask.unsqueeze(-1)

      return temporal_embeddings


model = GRU_Pipeline(
    input_size  = F,
    hidden_size = HIDDEN_SIZE,
    n_layers    = 2,
    dropout     = 0.3
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Paramètres totaux : {total_params:,}")
print(model)


In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 4 — ENTRAÎNEMENT
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ÉTAPE 4 — Entraînement")
print("=" * 60)

# Gestion du déséquilibre de classes
pos_weight = torch.tensor(
    [(y_arr == 0).sum() / (y_arr == 1).sum()],
    dtype=torch.float32
).to(device)
print(f"pos_weight (déséquilibre) : {pos_weight.item():.3f}")

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=LR, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=5,
    factor=0.5
)
current_lr = optimizer.param_groups[0]['lr']
print(f"LR actuel : {current_lr:.6f}")

best_val_auc = 0.0
best_weights = None
patience = 10
patience_counter = 0
history = {
    'train_loss': [], 'val_loss': [],
    'val_auc': [],    'val_f1': []
}


for epoch in range(1, N_EPOCHS + 1):

    # ── TRAIN ─────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for X_batch, y_batch, mask_batch in train_loader:
        X_batch    = X_batch.to(device)
        y_batch    = y_batch.to(device).unsqueeze(1)
        mask_batch = mask_batch.to(device)

        optimizer.zero_grad()
        _, risk_score = model(X_batch, mask_batch)

        loss = criterion(risk_score, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── VALIDATION ────────────────────────────────────────
    model.eval()
    val_loss  = 0.0
    val_preds = []
    val_true  = []

    with torch.no_grad():
        for X_batch, y_batch, mask_batch in val_loader:
            X_batch    = X_batch.to(device)
            y_batch    = y_batch.to(device).unsqueeze(1)
            mask_batch = mask_batch.to(device)

            _, risk_score = model(X_batch, mask_batch)
            loss = criterion(risk_score, y_batch)
            val_loss += loss.item()

            val_preds.extend(risk_score.cpu().numpy().flatten())
            val_true.extend(y_batch.cpu().numpy().flatten())

    val_loss /= len(val_loader)
    val_auc   = roc_auc_score(val_true, val_preds)

    # F1 au seuil 0.5
    val_pred_labels = (np.array(val_preds) >= 0.5).astype(int)
    val_true_arr    = np.array(val_true).astype(int)
    tp = ((val_pred_labels == 1) & (val_true_arr == 1)).sum()
    fp = ((val_pred_labels == 1) & (val_true_arr == 0)).sum()
    fn = ((val_pred_labels == 0) & (val_true_arr == 1)).sum()
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    val_f1 = 2 * prec * rec / (prec + rec + 1e-8)

    scheduler.step(val_auc)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_f1'].append(val_f1)

    # ── Early stopping + sauvegarde meilleur modèle ─────────────
    if val_auc > best_val_auc:
      best_val_auc = val_auc
      best_weights = {k: v.clone().detach().cpu()
                      for k, v in model.state_dict().items()}
      torch.save(best_weights, OUTPUT_PATH + 'best_gru_model1.pt')
      patience_counter = 0
    else:
      patience_counter += 1

    if patience_counter >= patience:
      print(f"Early stopping à l'epoch {epoch}")
      break

    if epoch % 5 == 0 or epoch == 1:
        print(f"Ep {epoch:3d}/{N_EPOCHS} | "
              f"Train Loss : {train_loss:.4f} | "
              f"Val Loss : {val_loss:.4f} | "
              f"Val AUC : {val_auc:.4f} | "
              f"Val F1 : {val_f1:.4f}"
              + (" ★ best" if val_auc == best_val_auc else ""))

# Charger les meilleurs poids
model.load_state_dict({k: v.to(device) for k, v in best_weights.items()})
print(f"\nMeilleur Val AUC : {best_val_auc:.4f}")

# ── Courbes d'entraînement ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Courbes d'entraînement — GRU", fontweight='bold')

axes[0].plot(history['train_loss'], label='Train', color='#2563EB')
axes[0].plot(history['val_loss'],   label='Val',   color='#DC2626')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['val_auc'], color='#7C3AED')
axes[1].set_title('AUC Validation'); axes[1].set_xlabel('Epoch')
axes[1].axhline(y=best_val_auc, color='red', ls='--', alpha=0.6,
                label=f'Best : {best_val_auc:.4f}')
axes[1].legend()

axes[2].plot(history['val_f1'], color='#16A34A')
axes[2].set_title('F1 Validation'); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.show()


In [ ]:
with open(OUTPUT_PATH + 'gru_training_history.pkl', 'wb') as f:
    pickle.dump(history, f)

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 5 — EVALUATION SUR LE TEST SET
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ÉTAPE 5 — Évaluation M3 (Test Set)")
print("=" * 60)

model.eval()
test_preds = []
test_true  = []

with torch.no_grad():
    for X_batch, y_batch, mask_batch in test_loader:
        X_batch    = X_batch.to(device)
        mask_batch = mask_batch.to(device)
        _, risk_score = model(X_batch, mask_batch)
        test_preds.extend(risk_score.cpu().numpy().flatten())
        test_true.extend(y_batch.numpy().flatten())

test_preds  = np.array(test_preds)
test_true   = np.array(test_true)
test_labels = (test_preds >= 0.5).astype(int)
test_auc    = roc_auc_score(test_true, test_preds)

print(f"Test AUC : {test_auc:.4f}")
print(f"\nRapport de classification :")
print(classification_report(
    test_true, test_labels,
    target_names=['Non à risque', 'À risque']
))

# Courbe ROC
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

fpr, tpr, _ = roc_curve(test_true, test_preds)
axes[0].plot(fpr, tpr, color='#7C3AED', lw=2,
             label=f'GRU (AUC = {test_auc:.4f})')
axes[0].plot([0,1],[0,1], 'k--', alpha=0.4)
axes[0].set_xlabel('Taux faux positifs')
axes[0].set_ylabel('Taux vrais positifs')
axes[0].set_title('Courbe ROC — Test Set', fontweight='bold')
axes[0].legend()

# Matrice de confusion
cm = confusion_matrix(test_true, test_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Non à risque', 'À risque'],
            yticklabels=['Non à risque', 'À risque'])
axes[1].set_title('Matrice de confusion', fontweight='bold')
axes[1].set_ylabel('Réel')
axes[1].set_xlabel('Prédit')

plt.tight_layout()
plt.show()

In [ ]:
# ── Sauvegarde risk scores ─────────────────────────────

gru_test_results = pd.DataFrame({
    'y_true': test_true,
    'risk_score': test_preds,
    'pred_label': test_labels
})

gru_test_results.to_csv(
    OUTPUT_PATH + 'gru_full_test_risk_scores.csv',
    index=False
)

print("\nRisk scores sauvegardés")

In [ ]:
gru_metrics = {
    'best_val_auc': float(best_val_auc),
    'test_auc': float(test_auc),
}

In [ ]:
print(np.mean(test_preds))
print(np.mean(test_labels))

# ════════════════════════════════════════════════════════════
# ETAPE 6 — EARLY WARNING PAR CHECKPOINT
# Un modèle GRU entraîné séparément pour chaque horizon
# ════════════════════════════════════════════════════════════

In [ ]:
from sklearn.metrics import roc_curve

print("\n" + "=" * 60)
print("ÉTAPE 6 — Early Warning par checkpoint")
print("=" * 60)

CHECKPOINTS = [3, 7, 12]
checkpoint_results = {}

for cp in CHECKPOINTS:
    print("\n" + "=" * 60)
    print(f"GRU Early Warning — Semaine {cp}")
    print("=" * 60)

    # 1. Couper les séquences jusqu'au checkpoint
    X_train_cp = X_train[:, :cp, :]
    X_val_cp   = X_val[:, :cp, :]
    X_test_cp  = X_test[:, :cp, :]

    mask_train_cp = mask_train[:, :cp]
    mask_val_cp   = mask_val[:, :cp]
    mask_test_cp  = mask_test[:, :cp]

    # 2. Datasets / DataLoaders
    train_loader_cp = DataLoader(
        OULADDataset(X_train_cp, y_train, mask_train_cp),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_loader_cp = DataLoader(
        OULADDataset(X_val_cp, y_val, mask_val_cp),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader_cp = DataLoader(
        OULADDataset(X_test_cp, y_test, mask_test_cp),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # 3. Nouveau modèle pour ce checkpoint
    model_cp = GRU_Pipeline(
        input_size=F,
        hidden_size=HIDDEN_SIZE,
        n_layers=2,
        dropout=0.3
    ).to(device)

    criterion = nn.BCELoss()

    optimizer = torch.optim.Adam(
        model_cp.parameters(),
        lr=LR,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        patience=5,
        factor=0.5
    )

    best_val_auc = 0.0
    best_weights = None

    history_cp = {
        'train_loss': [],
        'val_auc': []
    }

    # 4. Entraînement
    for epoch in range(1, N_EPOCHS + 1):
        model_cp.train()
        train_loss = 0.0

        for X_batch, y_batch, mask_batch in train_loader_cp:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)
            mask_batch = mask_batch.to(device)

            optimizer.zero_grad()

            _, risk_score = model_cp(X_batch, mask_batch)

            loss = criterion(risk_score, y_batch)
            loss.backward()

            nn.utils.clip_grad_norm_(model_cp.parameters(), max_norm=1.0)

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader_cp)

        # Validation
        model_cp.eval()
        val_preds = []
        val_true = []

        with torch.no_grad():
            for X_batch, y_batch, mask_batch in val_loader_cp:
                X_batch = X_batch.to(device)
                mask_batch = mask_batch.to(device)

                _, risk_score = model_cp(X_batch, mask_batch)

                val_preds.extend(risk_score.cpu().numpy().flatten())
                val_true.extend(y_batch.numpy().flatten())

        val_preds = np.array(val_preds)
        val_true = np.array(val_true)

        val_auc = roc_auc_score(val_true, val_preds)

        scheduler.step(val_auc)

        history_cp['train_loss'].append(train_loss)
        history_cp['val_auc'].append(val_auc)

        # Sauvegarder meilleur modèle
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_weights = {
                k: v.clone() for k, v in model_cp.state_dict().items()
            }

            torch.save(
                best_weights,
                OUTPUT_PATH + f'best_gru_checkpoint_w{cp}1.pt'
            )

        if epoch % 5 == 0 or epoch == 1:
            current_lr = optimizer.param_groups[0]['lr']
            print(
                f"Ep {epoch:02d}/{N_EPOCHS} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val AUC: {val_auc:.4f} | "
                f"LR: {current_lr:.6f}"
                + (" ★ best" if val_auc == best_val_auc else "")
            )

    # 5. Charger meilleur modèle
    model_cp.load_state_dict(best_weights)

    # 6. Test
    model_cp.eval()
    test_preds = []
    test_true = []

    with torch.no_grad():
        for X_batch, y_batch, mask_batch in test_loader_cp:
            X_batch = X_batch.to(device)
            mask_batch = mask_batch.to(device)

            _, risk_score = model_cp(X_batch, mask_batch)

            test_preds.extend(risk_score.cpu().numpy().flatten())
            test_true.extend(y_batch.numpy().flatten())

    test_preds = np.array(test_preds)
    test_true = np.array(test_true)
    test_labels = (test_preds >= 0.5).astype(int)

    test_auc = roc_auc_score(test_true, test_preds)

    tp = ((test_labels == 1) & (test_true == 1)).sum()
    fp = ((test_labels == 1) & (test_true == 0)).sum()
    fn = ((test_labels == 0) & (test_true == 1)).sum()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    checkpoint_results[cp] = {
        'model': model_cp,
        'best_val_auc': best_val_auc,
        'true'  : test_true,
        'preds': test_preds,
        'labels': test_labels,
        'auc': test_auc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'history': history_cp

    }

    print(f"\nRésultat Test — Semaine {cp}")
    print(f"AUC       : {test_auc:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1        : {f1:.4f}")


    checkpoint_preds_df = pd.DataFrame({
        'y_true': test_true,
        'y_pred_proba': test_preds,
        'y_pred_label': test_labels
        })


# ════════════════════════════════════════════════════════════
# Graphique comparaison checkpoints
# ════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(10, 5))

metrics = ['auc', 'recall', 'precision', 'f1']
colors_m = ['#7C3AED', '#DC2626', '#2563EB', '#16A34A']

x = np.arange(len(CHECKPOINTS))
width = 0.2

for j, (metric, color) in enumerate(zip(metrics, colors_m)):
    vals = [checkpoint_results[cp][metric] for cp in CHECKPOINTS]
    ax.bar(
        x + j * width,
        vals,
        width,
        label=metric.upper(),
        color=color,
        alpha=0.85
    )

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([f'Semaine {cp}' for cp in CHECKPOINTS])
ax.set_ylabel('Score')
ax.set_title(
    'Performance du GRU par checkpoint — Early Warning',
    fontweight='bold'
)
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()


fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Courbes d'entraînement par checkpoint", fontweight='bold')

for i, cp in enumerate(CHECKPOINTS):
    history = checkpoint_results[cp]['history']
    axes[i].plot(history['train_loss'], label='Train Loss',
                 color='#2563EB')
    ax2 = axes[i].twinx()
    ax2.plot(history['val_auc'], label='Val AUC',
             color='#DC2626', ls='--')
    axes[i].set_title(f'Semaine {cp}', fontweight='bold')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss', color='#2563EB')
    ax2.set_ylabel('AUC', color='#DC2626')
    axes[i].legend(loc='upper left')
    ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()



fig, ax = plt.subplots(figsize=(8, 6))
colors_cp = ['#DC2626', '#D97706', '#16A34A']

for cp, color in zip(CHECKPOINTS, colors_cp):
    fpr, tpr, _ = roc_curve(
        checkpoint_results[cp]['true'],
        checkpoint_results[cp]['preds']
    )
    auc = checkpoint_results[cp]['auc']
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'Semaine {cp} (AUC = {auc:.4f})')

ax.plot([0,1],[0,1], 'k--', alpha=0.4, label='Aléatoire')
ax.set_xlabel('Taux faux positifs')
ax.set_ylabel('Taux vrais positifs')
ax.set_title('Courbes ROC — Early Warning par checkpoint',
             fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ════════════════════════════════════════════════════════════
# Résumé tabulaire
# ════════════════════════════════════════════════════════════

checkpoint_summary = pd.DataFrame([
    {
        'checkpoint': cp,
        'best_val_auc': checkpoint_results[cp]['best_val_auc'],
        'test_auc': checkpoint_results[cp]['auc'],
        'precision': checkpoint_results[cp]['precision'],
        'recall': checkpoint_results[cp]['recall'],
        'f1': checkpoint_results[cp]['f1']
    }
    for cp in CHECKPOINTS
])

display(checkpoint_summary)


# ════════════════════════════════════════════════════════════
# Optimisation du seuil — Semaine 12
# ════════════════════════════════════════════════════════════

from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score
)

print("\n" + "=" * 60)
print("Optimisation du seuil — Semaine 12")
print("=" * 60)

thresholds = np.arange(0.30, 0.71, 0.05)

threshold_results = []

for thresh in thresholds:

    preds = (
        checkpoint_results[12]['preds'] >= thresh
    ).astype(int)

    rec = recall_score(
        checkpoint_results[12]['true'],
        preds
    )

    prec = precision_score(
        checkpoint_results[12]['true'],
        preds
    )

    f1 = f1_score(
        checkpoint_results[12]['true'],
        preds
    )

    threshold_results.append({
        'threshold': thresh,
        'recall': rec,
        'precision': prec,
        'f1': f1
    })

    print(
        f"Seuil={thresh:.2f} | "
        f"Recall={rec:.4f} | "
        f"Precision={prec:.4f} | "
        f"F1={f1:.4f}"
    )

threshold_df = pd.DataFrame(threshold_results)

print("\nRésumé optimisation seuil :")
display(threshold_df.round(4))

In [ ]:
# ── Seuil optimal retenu ──────────────────────────────────────
OPTIMAL_THRESHOLD = 0.35

print("=" * 60)
print("SEUIL OPTIMAL RETENU")
print("=" * 60)

for cp in CHECKPOINTS:
    preds_opt = (
        checkpoint_results[cp]['preds'] >= OPTIMAL_THRESHOLD
    ).astype(int)
    true      = checkpoint_results[cp]['true']

    tp = ((preds_opt==1) & (true==1)).sum()
    fp = ((preds_opt==1) & (true==0)).sum()
    fn = ((preds_opt==0) & (true==1)).sum()

    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    auc  = checkpoint_results[cp]['auc']

    print(f"\nSemaine {cp:2d} (seuil={OPTIMAL_THRESHOLD}) :")
    print(f"  AUC       : {auc:.4f}")
    print(f"  Recall    : {rec:.4f}  ← priorité EWS")
    print(f"  Precision : {prec:.4f}")
    print(f"  F1        : {f1:.4f}")

# Tableau final comparatif
print("\n" + "=" * 60)
print("TABLEAU FINAL — Architecture hybride complète")
print("=" * 60)

summary = pd.DataFrame([
    {
        'Modèle'   : 'XGBoost (Jour 0)',
        'Horizon'  : 'Avant le cours',
        'AUC'      : 0.6763,
        'Recall'   : 0.6199,
        'F1'       : 0.6380,
        'Seuil'    : 0.50,
    },
    {
        'Modèle'   : 'GRU (sem. 3)',
        'Horizon'  : 'Semaine 3',
        'AUC'      : 0.7781,
        'Recall'   : None,  # à calculer avec seuil optimal
        'F1'       : None,
        'Seuil'    : OPTIMAL_THRESHOLD,
    },
    {
        'Modèle'   : 'GRU (sem. 7)',
        'Horizon'  : 'Semaine 7',
        'AUC'      : 0.8542,
        'Recall'   : None,
        'F1'       : None,
        'Seuil'    : OPTIMAL_THRESHOLD,
    },
    {
        'Modèle'   : 'GRU (sem. 12)',
        'Horizon'  : 'Semaine 12',
        'AUC'      : 0.8948,
        'Recall'   : 0.833,   # avec seuil=0.40
        'F1'       : 0.816,
        'Seuil'    : OPTIMAL_THRESHOLD,
    },
    {
        'Modèle'   : 'GRU (complet)',
        'Horizon'  : '38 semaines',
        'AUC'      : 0.9774,
        'Recall'   : 0.910,
        'F1'       : 0.920,
        'Seuil'    : 0.50,
    },
])

display(summary)

In [ ]:
# ════════════════════════════════════════════════════════════
# SAUVEGARDE FINALE — EWS AVEC SEUIL OPTIMISÉ
# ════════════════════════════════════════════════════════════

OPTIMAL_THRESHOLD = 0.35

print("\n" + "=" * 60)
print("SAUVEGARDE — Résultats seuil optimisé")
print("=" * 60)

summary_rows = []

for cp in CHECKPOINTS:

    # ── Données checkpoint ──────────────────────────────────
    y_true      = checkpoint_results[cp]['true']
    risk_scores = checkpoint_results[cp]['preds']

    # ── Labels avec seuil optimisé ─────────────────────────
    pred_labels = (
        risk_scores >= OPTIMAL_THRESHOLD
    ).astype(int)

    # ── Métriques ──────────────────────────────────────────
    auc = roc_auc_score(y_true, risk_scores)

    precision = precision_score(y_true, pred_labels)
    recall    = recall_score(y_true, pred_labels)
    f1        = f1_score(y_true, pred_labels)

    # ── Affichage ──────────────────────────────────────────
    print(f"\nSemaine {cp} (seuil={OPTIMAL_THRESHOLD})")
    print(f"  AUC       : {auc:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  F1        : {f1:.4f}")

    # ── CSV predictions ────────────────────────────────────
    checkpoint_preds_df = pd.DataFrame({
        'y_true'     : y_true,
        'risk_score' : risk_scores,
        'pred_label' : pred_labels
    })

    checkpoint_preds_df.to_csv(
        OUTPUT_PATH + f'gru_checkpoint_w{cp}_predictions.csv',
        index=False
    )

    # ── Metrics dict ───────────────────────────────────────
    checkpoint_metrics = {
        'checkpoint'         : cp,
        'decision_threshold' : OPTIMAL_THRESHOLD,
        'best_val_auc'       : float(
            checkpoint_results[cp]['best_val_auc']
        ),
        'test_auc'           : float(auc),
        'precision'          : float(precision),
        'recall'             : float(recall),
        'f1'                 : float(f1),
    }

    with open(
        OUTPUT_PATH + f'gru_checkpoint_w{cp}_metrics.pkl',
        'wb'
    ) as f:
        pickle.dump(checkpoint_metrics, f)

    # ── Historique entraînement ────────────────────────────
    with open(
        OUTPUT_PATH + f'gru_checkpoint_w{cp}_history.pkl',
        'wb'
    ) as f:
        pickle.dump(
            checkpoint_results[cp]['history'],
            f
        )

    # ── Résumé global ──────────────────────────────────────
    summary_rows.append({
        'checkpoint'         : cp,
        'decision_threshold' : OPTIMAL_THRESHOLD,
        'best_val_auc'       : checkpoint_results[cp]['best_val_auc'],
        'test_auc'           : auc,
        'precision'          : precision,
        'recall'             : recall,
        'f1'                 : f1
    })

# ════════════════════════════════════════════════════════════
# Sauvegarde résumé global
# ════════════════════════════════════════════════════════════

checkpoint_summary = pd.DataFrame(summary_rows)

checkpoint_summary.to_csv(
    OUTPUT_PATH + 'gru_ews_checkpoint_summary.csv',
    index=False
)

print("\nRésumé checkpoints :")
display(checkpoint_summary.round(4))

# ════════════════════════════════════════════════════════════
# Sauvegarde risk scores
# ════════════════════════════════════════════════════════════

risk_scores_cp = {
    cp: checkpoint_results[cp]['preds']
    for cp in CHECKPOINTS
}

with open(
    OUTPUT_PATH + 'gru_checkpoint_risk_scores.pkl',
    'wb'
) as f:
    pickle.dump(risk_scores_cp, f)

print("\nRisk scores sauvegardés")

print("\n" + "=" * 60)
print("SAUVEGARDE TERMINÉE")
print("=" * 60)

# ════════════════════════════════════════════════════════════
# ETAPE 7 — EXTRACTION DES EMBEDDINGS
# ════════════════════════════════════════════════════════════

In [ ]:
print("\n" + "=" * 60)
print("ÉTAPE 7 — Extraction des embeddings")
print("=" * 60)

model.eval()

# ── Embeddings séquence complète ────────────────────────────
# Convertir en tensors
X_tensor = torch.tensor(X_seq, dtype=torch.float32)
mask_tensor = torch.tensor(mask_seq, dtype=torch.float32)

all_embeddings  = []
all_risk_scores = []

with torch.no_grad():
    for i in range(0, N, BATCH_SIZE):
        X_b    = X_tensor[i:i+BATCH_SIZE].to(device)
        mask_b = mask_tensor[i:i+BATCH_SIZE].to(device)
        emb, risk = model(X_b, mask_b)
        all_embeddings.append(emb.cpu().numpy())
        all_risk_scores.append(risk.cpu().numpy().flatten())

embeddings_full  = np.vstack(all_embeddings)    # (N, 64)
risk_scores_full = np.concatenate(all_risk_scores) # (N,)

np.save(
    OUTPUT_PATH + 'gru_embeddings_full.npy',
    embeddings_full
)

print(f"Embeddings complets : {embeddings_full.shape}")

np.save(
    OUTPUT_PATH + 'gru_risk_scores_full.npy',
    risk_scores_full
)

risk_df_full = pd.DataFrame({
    'risk_score': risk_scores_full
})

risk_df_full.to_csv(
    OUTPUT_PATH + 'gru_risk_scores_full.csv',
    index=False
)



# ════════════════════════════════════════════════════════════
# ETAPE 1 — EXTRACTION DES EMBEDDINGS TEMPORELS
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("ÉTAPE 1 — Extraction embeddings temporels (N, 38, 64)")
print("=" * 60)

model.eval()
X_tensor    = torch.tensor(X_seq,    dtype=torch.float32)
mask_tensor = torch.tensor(mask_seq, dtype=torch.float32)

temporal_embeddings_list = []

with torch.no_grad():
    for i in range(0, N, BATCH_SIZE):
        X_b    = X_tensor[i:i+BATCH_SIZE].to(device)
        mask_b = mask_tensor[i:i+BATCH_SIZE].to(device)

        temp_emb = model.get_temporal_embeddings(X_b, mask_b)
        temporal_embeddings_list.append(temp_emb.cpu().numpy())

# Shape finale : (N, 38, 64)
temporal_embeddings = np.vstack(temporal_embeddings_list)


# Sauvegarder
np.save(
    os.path.join(OUTPUT_PATH, 'temporal_embeddings.npy'),
    temporal_embeddings
)
print("Embeddings temporels sauvegardés")

print(f"Shape embeddings temporels : {temporal_embeddings.shape}")
print(f"  N étudiants : {temporal_embeddings.shape[0]:,}")
print(f"  T semaines  : {temporal_embeddings.shape[1]}")
print(f"  D dimensions: {temporal_embeddings.shape[2]}")

In [ ]:
# Embeddings par checkpoint
embeddings_cp  = {}
risk_scores_cp = {}

for cp in CHECKPOINTS:
    model_cp = checkpoint_results[cp]['model']
    model_cp.eval()

    X_cp = X_seq[:, :cp, :]
    mask_cp = mask_seq[:, :cp]

    X_tensor_cp = torch.tensor(X_cp, dtype=torch.float32)
    mask_tensor_cp = torch.tensor(mask_cp, dtype=torch.float32)

    emb_list = []
    risk_list = []

    with torch.no_grad():
        for i in range(0, len(X_cp), BATCH_SIZE):
            X_b = X_tensor_cp[i:i+BATCH_SIZE].to(device)
            mask_b = mask_tensor_cp[i:i+BATCH_SIZE].to(device)

            emb, risk = model_cp(X_b, mask_b)

            emb_list.append(emb.cpu().numpy())
            risk_list.append(risk.cpu().numpy().flatten())

    embeddings_cp[cp] = np.vstack(emb_list)
    risk_scores_cp[cp] = np.concatenate(risk_list)

    print(f"Semaine {cp} → embeddings {embeddings_cp[cp].shape} | "
          f"risk moyen : {risk_scores_cp[cp].mean():.3f}")


with open(
    OUTPUT_PATH + 'gru_embeddings_checkpoints.pkl', 'wb') as f: pickle.dump(embeddings_cp, f)

with open(
    OUTPUT_PATH + 'gru_risk_scores_checkpoints.pkl','wb') as f:pickle.dump(risk_scores_cp, f)

In [ ]:
embedding_metadata = {
    'embedding_dim': embeddings_full.shape[1],
    'n_students': embeddings_full.shape[0],
    'n_weeks': temporal_embeddings.shape[1],
    'checkpoints': CHECKPOINTS,
}

with open(
    OUTPUT_PATH + 'gru_embedding_metadata.pkl',
    'wb'
) as f:
    pickle.dump(embedding_metadata, f)


student_ids_df = final_df_wcdmacp[
    student_id_cols
].reset_index(drop=True)

student_ids_df.to_csv(
    OUTPUT_PATH + 'student_ids_embeddings.csv',
    index=False
)